# LingMate — бесплатный QLoRA-пайплайн на Colab (TRL ≥ 0.22)

*Автоматически собранный чистый ноутбук.*  
Дата: **2025-08-09 22:01:22**  
Особенности:
- Совместим с новой TRL (0.22+): использует `processing_class` + `formatting_func`
- Сохраняет чекпойнты в Google Диск
- Отключён Weights & Biases (wandb)
- Чёткая структура с русскими заголовками

## 1. Проверка GPU

In [1]:
!nvidia-smi

Sat Aug  9 22:04:16 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Подключить Google Диск

In [2]:
from google.colab import drive
drive.mount('/content/drive')
WORKDIR = '/content/drive/MyDrive/LingMateColab'
import os
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
print("Рабочая папка:", WORKDIR)

Mounted at /content/drive
Рабочая папка: /content/drive/MyDrive/LingMateColab


## 3. Установка зависимостей + вывод версий

In [5]:
!pip -q install --upgrade pip
!pip -q install "transformers>=4.43.3" "datasets>=2.20.0" "accelerate>=0.33.0" \
                "peft>=0.11.1" bitsandbytes sentencepiece evaluate tiktoken huggingface_hub
# TRL — ставим «самую свежую» из исходников (main)
!pip -q install git+https://github.com/huggingface/trl.git

import torch, transformers, datasets, peft, trl, platform
print("Python:", platform.python_version())
print("Torch:", torch.__version__, "CUDA:", torch.cuda.is_available())
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Python: 3.11.13
Torch: 2.6.0+cu124 CUDA: True
Transformers: 4.55.0
Datasets: 4.0.0
PEFT: 0.17.0
TRL: 0.22.0.dev0


## 4. Скрипт обучения (QLoRA) — совместим с TRL ≥ 0.22

In [6]:
%%writefile train_lora.py
import os
os.environ["WANDB_DISABLED"] = "true"  # отключить wandb

import argparse
from dataclasses import dataclass
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

SYSTEM_PROMPT = (
    "You are LingMate, a friendly, evidence-based language coach. "
    "Teach using short, clear steps, examples from real-life media, and spaced repetition hints. "
    "Adapt to the learner's level and never overwhelm."
)

def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument('--data_path', type=str, required=True)
    p.add_argument('--base_model', type=str, default='Qwen/Qwen2.5-3B-Instruct')
    p.add_argument('--output_dir', type=str, default='checkpoints/qwen2p5-3b-lingmate')
    p.add_argument('--max_len', type=int, default=512)
    p.add_argument('--epochs', type=int, default=1)
    p.add_argument('--lr', type=float, default=2e-4)
    p.add_argument('--batch', type=int, default=1)
    p.add_argument('--grad_accum', type=int, default=32)
    p.add_argument('--warmup_ratio', type=float, default=0.03)
    p.add_argument('--lora_r', type=int, default=16)
    p.add_argument('--lora_alpha', type=int, default=32)
    p.add_argument('--lora_dropout', type=float, default=0.05)
    p.add_argument('--no_4bit', action='store_true')
    return p.parse_args()

from dataclasses import dataclass
@dataclass
class Rec:
    lang: str
    source: str
    segment_type: str
    content: str

def make_chat(tokenizer, rec: Rec):
    user = (
        f"Learner language: {rec.lang}. Source={rec.source}, type={rec.segment_type}.\\n"
        f"Please teach me using a short, kind explanation (+ 1-2 examples)."
    )
    assistant = rec.content.strip()
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user},
        {"role": "assistant", "content": assistant},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

def main():
    args = parse_args()
    compute_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

    bnb_config = None
    if not args.no_4bit:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.bfloat16,
        )

    tokenizer = AutoTokenizer.from_pretrained(args.base_model, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        args.base_model,
        torch_dtype=compute_dtype,
        device_map='auto',
        quantization_config=bnb_config,
    )

    if bnb_config is not None:
        model = prepare_model_for_kbit_training(model)
        if hasattr(model, 'gradient_checkpointing_enable'):
            try:
                model.gradient_checkpointing_enable()
            except Exception:
                pass

    lora = LoraConfig(
        r=args.lora_r,
        lora_alpha=args.lora_alpha,
        lora_dropout=args.lora_dropout,
        bias='none',
        task_type='CAUSAL_LM',
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    )
    model = get_peft_model(model, lora)

    raw = load_dataset('json', data_files=args.data_path, split='train')

    def map_to_text(batch):
        texts = []
        for lang, source, stype, content in zip(
            batch['lang'], batch['source'], batch['segment_type'], batch['content']
        ):
            rec = Rec(lang, source, stype, content)
            texts.append(make_chat(tokenizer, rec))
        return {'text': texts}

    ds = raw.map(map_to_text, batched=True, remove_columns=raw.column_names)

    def format_batch(example):
        return example["text"]

    config = SFTConfig(
        output_dir=args.output_dir,
        do_train=True,
        num_train_epochs=args.epochs,
        per_device_train_batch_size=args.batch,
        gradient_accumulation_steps=args.grad_accum,
        learning_rate=args.lr,
        lr_scheduler_type='cosine',
        warmup_ratio=args.warmup_ratio,
        logging_steps=10,
        save_steps=200,
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        args=config,
        train_dataset=ds,
        processing_class=tokenizer,
        formatting_func=format_batch,
    )
    trainer.train()
    trainer.save_model(args.output_dir)
    tokenizer.save_pretrained(args.output_dir)

if __name__ == '__main__':
    main()

Overwriting train_lora.py


## 5. Мини-датасет для дымового теста

In [16]:
import os, json
os.makedirs('data/processed', exist_ok=True)
sample = [
    {"lang":"en","source":"textbook","segment_type":"explanation","content":"Present simple is used for habits and facts. Example: I drink coffee every morning. Fact: Water boils at 100°C."},
    {"lang":"es","source":"youtube","segment_type":"dialogue","content":"¿Me pone un café, por favor? —Claro, ¿algo más? —No, gracias."},
    {"lang":"fr","source":"textbook","segment_type":"example","content":"Je voudrais une baguette, s'il vous plaît. —Ça fait 1,20 €. —Tenez."},
    {"lang":"de","source":"textbook","segment_type":"explanation","content":"Artikel im Nominativ: der (m), die (f), das (n), die (Pl.). Beispiel: Der Mann ist müde."},
    {"lang":"ja","source":"podcast","segment_type":"dialogue","content":"すみません、駅はどこですか。—この道をまっすぐ行って、左です。"},
    {"lang":"it","source":"textbook","segment_type":"example","content":"Vorrei un gelato al pistacchio, per favore. —Subito!"},
    {"lang":"ko","source":"textbook","segment_type":"explanation","content":"조사의 기본: 은/는(주제), 이/가(주어). 예: 저는 학생이에요. 친구가 와요."},
    {"lang":"zh","source":"youtube","segment_type":"dialogue","content":"请问，地铁站在哪里？—在前面右转。"},
    {"lang":"pt","source":"textbook","segment_type":"example","content":"Eu gostaria de um copo de água, por favor. —Aqui está."},
    {"lang":"hi","source":"textbook","segment_type":"explanation","content":"हिंदी में पोस्टपोज़िशन संज्ञा के बाद आती हैं। उदाहरण: मेज़ के ऊपर (on the table)."},
    {"lang":"en","source":"podcast","segment_type":"explanation","content":"Spaced repetition: review new words after 1, 3, 7 days. Keep examples short and vivid."},
    {"lang":"es","source":"textbook","segment_type":"example","content":"Para pedir en un bar: ¿Me pone una tapa de tortilla? —Sí, ahora mismo."}
]
with open('data/processed/lingmate_sample.jsonl', 'w', encoding='utf-8') as f:
    for r in sample:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')
print('Wrote data/processed/lingmate_sample.jsonl (', len(sample), 'records)')

Wrote data/processed/lingmate_sample.jsonl ( 12 records)


## 6. Обучение (QLoRA на Qwen2.5-3B-Instruct)

In [17]:
import os
WORKDIR = '/content/drive/MyDrive/LingMateColab'
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)

OUTPUT_DIR = f"{WORKDIR}/checkpoints/qwen2p5-3b-lingmate"
DATA_PATH  = f"{WORKDIR}/data/processed/lingmate_sample.jsonl"
print('OUTPUT_DIR:', OUTPUT_DIR)
print('DATA_PATH:', DATA_PATH)

!python train_lora.py \
  --data_path "{DATA_PATH}" \
  --base_model Qwen/Qwen2.5-3B-Instruct \
  --output_dir "{OUTPUT_DIR}" \
  --epochs 1 --lr 2e-4 --batch 1 --grad_accum 32 --max_len 512

OUTPUT_DIR: /content/drive/MyDrive/LingMateColab/checkpoints/qwen2p5-3b-lingmate
DATA_PATH: /content/drive/MyDrive/LingMateColab/data/processed/lingmate_sample.jsonl
2025-08-09 22:25:06.928823: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754778306.949798    6014 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754778306.955916    6014 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1754778306.971309    6014 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1754778306.971338    6014 computation_placer.cc:177] computation placer already regi

## 7. Проверка чекпойнта

In [18]:
import os, glob
WORKDIR = '/content/drive/MyDrive/LingMateColab'
os.chdir(WORKDIR)
print('CWD =', os.getcwd())
print('Checkpoints dir exists:', os.path.exists('checkpoints'))
print('LingMate ckpt exists:', os.path.exists('checkpoints/qwen2p5-3b-lingmate'))
print('Files inside ckpt:', glob.glob('checkpoints/qwen2p5-3b-lingmate/*'))

CWD = /content/drive/MyDrive/LingMateColab
Checkpoints dir exists: True
LingMate ckpt exists: True
Files inside ckpt: ['checkpoints/qwen2p5-3b-lingmate/runs', 'checkpoints/qwen2p5-3b-lingmate/checkpoint-1', 'checkpoints/qwen2p5-3b-lingmate/README.md', 'checkpoints/qwen2p5-3b-lingmate/adapter_model.safetensors', 'checkpoints/qwen2p5-3b-lingmate/training_args.bin', 'checkpoints/qwen2p5-3b-lingmate/added_tokens.json', 'checkpoints/qwen2p5-3b-lingmate/tokenizer_config.json', 'checkpoints/qwen2p5-3b-lingmate/adapter_config.json', 'checkpoints/qwen2p5-3b-lingmate/chat_template.jinja', 'checkpoints/qwen2p5-3b-lingmate/special_tokens_map.json', 'checkpoints/qwen2p5-3b-lingmate/merges.txt', 'checkpoints/qwen2p5-3b-lingmate/vocab.json', 'checkpoints/qwen2p5-3b-lingmate/tokenizer.json']


## 8. Быстрый инференс (с аккуратным выводом только ответа)

In [19]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch, os

WORKDIR = '/content/drive/MyDrive/LingMateColab'
CKPT = f"{WORKDIR}/checkpoints/qwen2p5-3b-lingmate"
BASE = 'Qwen/Qwen2.5-3B-Instruct'

os.chdir(WORKDIR)
tokenizer = AutoTokenizer.from_pretrained(BASE)
base = AutoModelForCausalLM.from_pretrained(
    BASE, device_map='auto',
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32
)
model = PeftModel.from_pretrained(base, CKPT, local_files_only=True)

messages = [
    {"role": "system", "content": "You are LingMate, a friendly language coach. Keep answers short, accurate and idiomatic."},
    {"role": "user", "content": "Teach me how to order coffee politely in Spanish with 2-3 natural phrases and brief explanations in English."},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

with torch.no_grad():
    out = model.generate(
        **inputs, max_new_tokens=220, do_sample=False, temperature=0.3, top_p=0.9
    )

gen_tokens = out[0][inputs["input_ids"].shape[1]:]
print(tokenizer.decode(gen_tokens, skip_special_tokens=True).strip())

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Sure! Here are some polite ways to order coffee in Spanish:

1. **"¿Podría traerme una taza de café solo, por favor?"**
   - Explanation: This is a straightforward way to ask for a single cup of coffee. "Sí, por favor" (Yes, please) can be added at the end if needed.

2. **"¿Me podrías servir un café americano, por favor?"**
   - Explanation: This phrase asks for an Americano, which is espresso with hot water. It's a bit more specific but still polite.

3. **"¿Podría darme una taza de café con leche, por favor?"**
   - Explanation: This is a common request for a coffee with milk. Again, "sí, por favor" can be added for emphasis.

These phrases should help you get your coffee ordered politely in Spanish.


## 9. Ещё один пример (русская инструкция → испанские фразы)

In [20]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch, os

WORKDIR = '/content/drive/MyDrive/LingMateColab'
CKPT = f"{WORKDIR}/checkpoints/qwen2p5-3b-lingmate"
BASE = 'Qwen/Qwen2.5-3B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(BASE)
base = AutoModelForCausalLM.from_pretrained(
    BASE, device_map='auto',
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32
)
model = PeftModel.from_pretrained(base, CKPT, local_files_only=True)

messages = [
    {"role": "system", "content": "You are LingMate. Keep answers short and idiomatic. Correct mistakes silently."},
    {"role": "user", "content": "Дай 3 естественные фразы, как в Испании вежливо заказать кофе на вынос. Переведи на русский и добавь короткий совет по произношению."},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

with torch.no_grad():
    out = model.generate(
        **inputs, max_new_tokens=220, do_sample=False, temperature=0.3, top_p=0.9
    )

gen_tokens = out[0][inputs["input_ids"].shape[1]:]
print(tokenizer.decode(gen_tokens, skip_special_tokens=True).strip())

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


1. "¿Podría indicarme dónde puedo ordenar un café tostado por favor?"
Перевод: "Можете мне указать, где я могу заказать тостовый кофе, пожалуйста?"
Совет: "Постарайтесь произнести 'tostado' с акцентом на последний слог, чтобы звучало как 'то-ТАДО'."

2. "¿Me podrías decir dónde puedo conseguir una taza de café americano, por favor?"
Перевод: "Можете мне сказать, где я могу найти чашку американского кофе, пожалуйста?"
Совет: "В этом случае акцент делайте на слове 'americano', чтобы оно звучало как 'амерИКАНО'."

3. "¿Podría mostrarme el camino para llegar a la máquina expendedora de café, por favor?"
Перевод: "Можете мне показать путь


## 10. Подсказки по качеству
- Если ответы «плывут», уменьшить `temperature` до `0.2` и оставить `do_sample=False`.
- Добавлять **короткие, точные примеры** в датасет — это лучше всего улучшает поведение.
- Для стабильности на T4 держать `--max_len 512`, `--batch 1`, `--grad_accum 32`.